In [ ]:
import pandas as pd
from transformers import pipeline
from tqdm import tqdm
flores = pd.read_parquet('data/test_set/flores_plus.parquet')
wmt14 = pd.read_parquet('data/test_set/wmt14_test.parquet')
tqdm.pandas(desc="Progress")

In [ ]:
opus_translator = pipeline("translation_en_to_fr", model="Helsinki-NLP/opus-mt-en-fr")
wmt14['mt_opus'] = wmt14.en.progress_apply(lambda x:opus_translator(x)[0]['translation_text'])
flores['mt_opus'] = flores.text_eng.progress_apply(lambda x:opus_translator(x)[0]['translation_text'])

In [ ]:
# Load the T5 model using the pipeline
t5_translator = pipeline("text2text-generation", model="google-t5/t5-small")
wmt14['mt_t5'] = wmt14.en.progress_apply(lambda x:t5_translator(f"translate English to French: {x}", max_length=512)[0]['generated_text'])
flores['mt_t5'] = flores.text_eng.progress_apply(lambda x:t5_translator(f"translate English to French: {x}", max_length=512)[0]['generated_text'])


In [ ]:
# Export results
# wmt14.to_parquet('data/test_set/wmt14_mt_output.parquet', index=False)
# flores.to_parquet('data/test_set/flores_mt_output.parquet', index=False)

# Import results
flores_output = pd.read_parquet('data/test_set/flores_mt_output.parquet')
wmt14_output = pd.read_parquet('data/test_set/wmt14_mt_output.parquet')
# Convert to text files for scoring
wmt14_output.fr.to_csv("data/test_set/wmt14/ref.txt", index=False, header=False)
wmt14_output.en.to_csv("data/test_set/wmt14/src.txt", index=False, header=False)
wmt14_output.mt_opus.to_csv("data/test_set/wmt14/mt_opus.txt", index=False, header=False)
wmt14_output.mt_t5.to_csv("data/test_set/wmt14/mt_t5.txt", index=False, header=False)
flores_output.text_fra.to_csv("data/test_set/flores/ref.txt", index=False, header=False)
flores_output.text_eng.to_csv("data/test_set/flores/src.txt", index=False, header=False)
flores_output.mt_opus.to_csv("data/test_set/flores/mt_opus.txt", index=False, header=False)
flores_output.mt_t5.to_csv("data/test_set/flores/mt_t5.txt", index=False, header=False)